# Calibrating bias correction for a realistic Texas summer pool temp

**Goal**: find a `bias_correction` mode + parameter setting in
`scripts/pool_physics.py` that produces a *realistic* equilibrium pool
temperature for a hot, humid Texas summer scenario:

- Day high air temp  ≈ 110 °F (43.3 °C)
- Night low air temp ≈  80 °F (26.7 °C)
- Humid (RH ~55% day / ~80% night — Waco/Central-TX summer, not desert-dry)
- Target: pool settles around **~90 °F (32.2 °C)**, which matches typical
  observed wave-pool behavior at BSR Waco in peak summer — hot, but not
  air-temperature-hot, because a 2 m deep pool has a lot of thermal mass and
  strong evaporative cooling once wind/agitation is accounted for.

This follows on from `penman_bias_correction_sanity_check.ipynb`, which
showed a **single time-step** snapshot of each bias-correction mode. That's
useful for isolating flux math, but a single step can't show whether a mode
actually reaches a *sensible equilibrium* over many days — this notebook
does that by running a full multi-day diurnal simulation to a repeating
daily cycle (steady periodic state) and reports the resulting pool temp.


## Setup

In [1]:
import math

from pool_physics import (
    pool_thermal_balance_step,
    CE_MULTIPLIER,
    EVAP_WIND_FLOOR_MS,
)

def C_to_F(c):
    return c * 9 / 5 + 32

def F_to_C(f):
    return (f - 32) * 5 / 9

print(f"Production defaults: CE_MULTIPLIER={CE_MULTIPLIER}, EVAP_WIND_FLOOR_MS={EVAP_WIND_FLOOR_MS} m/s")

Production defaults: CE_MULTIPLIER=1.25, EVAP_WIND_FLOOR_MS=1.5 m/s


## Texas humid-summer diurnal forcing

Builds hourly `T_air`, `RH`, `wind`, and `solar` from smooth diurnal curves
anchored to the day-high / night-low targets, rather than a single flat
value — a single daily-mean step (as used in the earlier sanity-check
notebook and in `pull_api_data.py`'s daily model) washes out the fact that
the hottest, calmest, sunniest *hours* are what drive peak pool warming.
Real Waco pool config values (`scripts/pools_config.json`) are used for
`depth_m`, `ground_temp_C`, and the derived bottom U-value.

In [2]:
def diurnal_forcing(hour,
                     day_high_C=F_to_C(110), night_low_C=F_to_C(80),
                     rh_day=55.0, rh_night=80.0,
                     wind_day=2.6, wind_night=1.4,
                     solar_peak_Wm2=980.0):
    """Smooth diurnal cycle for one hour-of-day (0-24, fractional OK)."""
    frac = (hour - 6.0) / 24.0
    T_air = ((day_high_C + night_low_C) / 2
             + (day_high_C - night_low_C) / 2 * math.cos(2 * math.pi * (frac - 0.5)))

    # RH tracks inversely with temperature (dry heat of the day, humid at night)
    rh_frac = (T_air - night_low_C) / (day_high_C - night_low_C)
    RH = rh_day + (rh_night - rh_day) * (1 - rh_frac)

    # Wind picks up slightly at midday, calmer overnight
    if 6.0 <= hour <= 18.0:
        wind = wind_night + (wind_day - wind_night) * max(0.0, math.sin(math.pi * (hour - 6) / 12))
    else:
        wind = wind_night

    # Daylight solar bell curve, 6 AM - 8 PM (long Texas summer day)
    if 6.0 <= hour <= 20.0:
        solar_Wm2 = solar_peak_Wm2 * math.sin(math.pi * (hour - 6) / 14.0)
    else:
        solar_Wm2 = 0.0
    solar_MJm2_day_equiv = solar_Wm2 * 86400.0 / 1.0e6  # units pool_thermal_balance_step expects

    return T_air, RH, wind, solar_MJm2_day_equiv


# Sanity-check the forcing curve at a few hours
for h in [3, 9, 15, 21]:
    T_air, RH, wind, solar = diurnal_forcing(h)
    print(f"hour={h:2d}  T_air={C_to_F(T_air):5.1f}F  RH={RH:4.1f}%  wind={wind:.1f} m/s  solar={solar:.1f} MJ/m2/day-equiv")


hour= 3  T_air= 84.4F  RH=76.3%  wind=1.4 m/s  solar=0.0 MJ/m2/day-equiv
hour= 9  T_air= 84.4F  RH=76.3%  wind=2.2 m/s  solar=52.8 MJ/m2/day-equiv
hour=15  T_air=105.6F  RH=58.7%  wind=2.2 m/s  solar=76.3 MJ/m2/day-equiv
hour=21  T_air=105.6F  RH=58.7%  wind=1.4 m/s  solar=0.0 MJ/m2/day-equiv


## Real Waco pool physical parameters

From `scripts/pools_config.json` — `depth_m`, `ground_temp_C`, and a bottom
U-value derived the same way `pull_api_data.py` derives it (concrete slab +
soil column in series).

In [3]:
K_CONCRETE   = 1.4      # W/m/K (matches pool_physics.py)
L_CONCRETE_M = 0.3048   # 1 ft slab

# Waco config (scripts/pools_config.json)
depth_m       = 2.0
ground_temp_C = 20.0
k_soil_Wm1K   = 1.5
soil_depth_m  = 2.0

R_concrete = L_CONCRETE_M / K_CONCRETE
R_soil     = soil_depth_m / k_soil_Wm1K
bottom_u_Wm2K = 1.0 / (R_concrete + R_soil)

print(f"depth_m={depth_m}  ground_temp_C={ground_temp_C}  bottom_u_Wm2K={bottom_u_Wm2K:.3f}")


depth_m=2.0  ground_temp_C=20.0  bottom_u_Wm2K=0.645


## Multi-day diurnal simulation harness

Steps `pool_thermal_balance_step` hourly (`dt_days = 1/24`) for `days` days
starting from an arbitrary initial pool temp, and returns the **last day's**
hourly pool-temp series once the simulation has settled into a repeating
daily cycle (checked via `drift_C`, the max difference between the last two
days' hourly series — near zero means steady periodic state).

In [4]:
def run_sim(mode, days=20, T_pool_start_C=30.0,
            ce_multiplier=None, evap_wind_floor_ms=None):
    T_pool = T_pool_start_C
    dt = 1.0 / 24.0
    prev_day, last_day = None, None

    for _ in range(days):
        hourly = []
        for h in range(24):
            T_air, RH, wind, solar = diurnal_forcing(h + 0.5)
            T_pool, fx = pool_thermal_balance_step(
                T_pool, T_air, RH, wind, solar,
                depth_m=depth_m, ground_temp_C=ground_temp_C, bottom_u_Wm2K=bottom_u_Wm2K,
                dt_days=dt, bias_correction=mode,
                ce_multiplier=ce_multiplier, evap_wind_floor_ms=evap_wind_floor_ms,
            )
            hourly.append(T_pool)
        prev_day, last_day = last_day, hourly

    drift_C = max(abs(a - b) for a, b in zip(last_day, prev_day)) if prev_day else float('nan')
    return {
        'hourly_C': last_day,
        'mean_C': sum(last_day) / len(last_day),
        'min_C': min(last_day),
        'max_C': max(last_day),
        'drift_C': drift_C,
    }


## Step 1 — baseline behavior of all four modes (production defaults)

Using the *current* `pool_physics.py` defaults (`CE_MULTIPLIER=1.25`,
`EVAP_WIND_FLOOR_MS=1.5`) under sustained Texas humid-summer forcing —
not just a single hot/calm snapshot.

In [5]:
print(f"{'mode':16s} {'mean':>8s} {'min':>8s} {'max':>8s} {'drift_C':>10s}")
for mode in ['none', 'evap_multiplier', 'wind_floor', 'penman']:
    r = run_sim(mode)
    print(f"{mode:16s} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F {r['drift_C']:10.4f}")

print("\nTarget: mean ~90F, reflecting a large, well-agitated wave pool that")
print("stays much cooler than the 110F day-high air temp thanks to strong")
print("evaporative cooling.")


mode                 mean      min      max    drift_C
none               107.0F   105.4F   108.6F     0.0038
evap_multiplier    105.1F   103.5F   106.7F     0.0020
wind_floor         106.8F   105.1F   108.4F     0.0036
penman             122.1F   120.3F   123.7F     0.0876

Target: mean ~90F, reflecting a large, well-agitated wave pool that
stays much cooler than the 110F day-high air temp thanks to strong
evaporative cooling.


### Result

All four modes at their *default* parameters settle into an unrealistically
hot equilibrium (~103-107°F for `none`/`evap_multiplier`/`wind_floor`, and
`penman` is even hotter at ~121°F, consistent with the earlier notebook
finding that `penman` under-predicts cooling). None of these are close to
the ~90°F target — the baseline model (and the notebook's small `1.25x`/
`1.5 m/s` tuning, which was fit for a different, presumably milder scenario)
isn't strong enough evaporative cooling for a **110°F humid Texas
day**. `penman` remains the worst performer and still needs a separate fix
(see the previous notebook's "Suspected cause" section) — it is excluded
from further tuning here.

## Step 2 — sweep `evap_multiplier` and `wind_floor` magnitudes

Rather than guess, sweep both correction modes across a wide range of
magnitudes and see what it actually takes to reach ~90°F under this
forcing.

In [6]:
print("=== evap_multiplier (CE_MULTIPLIER) sweep ===")
print(f"{'CE_MULTIPLIER':>14s} {'mean':>8s} {'min':>8s} {'max':>8s}")
for cm in [1.25, 3, 5, 8, 10, 12, 16, 20]:
    r = run_sim('evap_multiplier', ce_multiplier=cm)
    print(f"{cm:14.2f} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F")

print()
print("=== wind_floor (EVAP_WIND_FLOOR_MS) sweep ===")
print(f"{'wind_floor_ms':>14s} {'mean':>8s} {'min':>8s} {'max':>8s}")
for wf in [1.5, 3, 5, 8, 10, 12, 16, 20]:
    r = run_sim('wind_floor', evap_wind_floor_ms=wf)
    print(f"{wf:14.1f} {C_to_F(r['mean_C']):7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F")


=== evap_multiplier (CE_MULTIPLIER) sweep ===
 CE_MULTIPLIER     mean      min      max
          1.25   105.1F   103.5F   106.7F
          3.00    97.6F    95.8F    99.2F
          5.00    93.6F    91.7F    95.4F
          8.00    90.7F    88.4F    92.6F
         10.00    89.5F    87.0F    91.6F
         12.00    88.6F    85.9F    90.9F
         16.00    87.5F    84.3F    90.2F
         20.00    86.7F    83.1F    89.8F

=== wind_floor (EVAP_WIND_FLOOR_MS) sweep ===
 wind_floor_ms     mean      min      max
           1.5   106.8F   105.1F   108.4F
           3.0   102.4F   100.5F   104.2F
           5.0    98.0F    96.0F    99.8F
           8.0    94.2F    92.0F    96.2F
          10.0    92.6F    90.3F    94.7F
          12.0    91.4F    89.0F    93.6F
          16.0    89.7F    87.1F    92.1F
          20.0    88.6F    85.7F    91.1F


### Result

Both modes can reach ~90°F, but only with parameter values **far beyond**
the notebook-derived defaults (`CE_MULTIPLIER=1.25`, `EVAP_WIND_FLOOR_MS=1.5`):

- `evap_multiplier` needs **CE_MULTIPLIER ≈ 8** (a 6-8x jump from 1.25) to
  land near 90°F.
- `wind_floor` needs **EVAP_WIND_FLOOR_MS ≈ 12-16 m/s** (vs. 1.5 m/s
  default) — i.e. treating the *whole pool surface* as if a steady
  12-16 m/s wind were blowing across it at all times.

Both of these are large extrapolations from the original notebook's
sensitivity sweep, which was fit against milder conditions. But they are
**not physically absurd** for this specific application: this is a **wave
pool**, not a placid backyard pool — a wave-generating machine constantly
agitates and aerates a large fraction of the surface, which is a much
stronger analogue to "high sustained wind" than the bulk-transfer
formula (calibrated for calm, undisturbed open water) assumes. A wind-floor
in the 12-16 m/s-equivalent range is a plausible stand-in for "surface
constantly disturbed by wave machinery," rather than natural wind alone.

## Step 3 — pick a specific calibration and confirm it holds up

In [7]:
chosen_mode = 'wind_floor'
chosen_wind_floor = 14.0

r = run_sim(chosen_mode, evap_wind_floor_ms=chosen_wind_floor)
print(f"mode={chosen_mode}  evap_wind_floor_ms={chosen_wind_floor}")
print(f"  mean = {C_to_F(r['mean_C']):.1f}F   min = {C_to_F(r['min_C']):.1f}F   max = {C_to_F(r['max_C']):.1f}F")
print(f"  drift_C (steady-state check) = {r['drift_C']:.5f}")
print()
print("Hourly pool temp over the final (settled) day:")
for h, t_c in enumerate(r['hourly_C']):
    marker = '  <-- night low' if h in (4, 5) else ('  <-- afternoon peak' if h in (15, 16) else '')
    print(f"  hour {h:2d}: {C_to_F(t_c):5.1f}F{marker}")


mode=wind_floor  evap_wind_floor_ms=14.0
  mean = 90.5F   min = 88.0F   max = 92.7F
  drift_C (steady-state check) = 0.00000

Hourly pool temp over the final (settled) day:
  hour  0:  91.6F
  hour  1:  91.1F
  hour  2:  90.6F
  hour  3:  90.0F
  hour  4:  89.4F  <-- night low
  hour  5:  88.8F  <-- night low
  hour  6:  88.3F
  hour  7:  88.0F
  hour  8:  88.0F
  hour  9:  88.1F
  hour 10:  88.4F
  hour 11:  88.8F
  hour 12:  89.4F
  hour 13:  90.1F
  hour 14:  90.7F
  hour 15:  91.4F  <-- afternoon peak
  hour 16:  92.0F  <-- afternoon peak
  hour 17:  92.4F
  hour 18:  92.7F
  hour 19:  92.7F
  hour 20:  92.6F
  hour 21:  92.5F
  hour 22:  92.3F
  hour 23:  92.0F


### Result

`wind_floor=14.0 m/s` settles into a steady daily cycle (`drift_C` ≈ 0)
averaging **~90°F**, ranging roughly **88-93°F** across the day — hot, but
meaningfully below the 110°F day-high / cooler than a placid pool would be,
and never dropping below the 80°F night-air floor. This matches the target
scenario description ("90 in humid 110 deg air, 80 nights").

## Conclusion / recommendation

- **Do not use `penman`** as currently implemented — it under-cools even
  worse than baseline and needs the fix identified in
  `penman_bias_correction_sanity_check.ipynb` before it's usable.
- **`evap_multiplier` (CE_MULTIPLIER≈8) or `wind_floor` (EVAP_WIND_FLOOR_MS≈14 m/s)**
  both reach a realistic ~90°F equilibrium for this scenario.
  `wind_floor` is preferred here because its physical interpretation (a
  wave-machine-agitated surface behaving like sustained high wind) maps
  more directly onto *why* a wave pool should evaporate/cool faster than
  the plain bulk-transfer formula assumes, vs. `evap_multiplier`'s more
  generic "just scale the coefficient" fit.
- These are **exploratory findings only** — `pool_physics.py`'s production
  defaults (`ACTIVE_BIAS_CORRECTION='none'`, `CE_MULTIPLIER=1.25`,
  `EVAP_WIND_FLOOR_MS=1.5`) have **not** been changed by this notebook.
  Flip `ACTIVE_BIAS_CORRECTION` (or pass `bias_correction=` explicitly at
  call sites) only after confirming this holds up against real observed
  pool-temperature data, not just a synthetic diurnal forcing curve.

## Appendix — Initial Conditions & Solver Reference

Everything above reports summary statistics. For reproducibility, this
appendix prints the exact starting conditions, a couple of days of the
forcing the solver actually consumed, and the full source of every
function involved in the simulation (`diurnal_forcing`,
`pool_thermal_balance_step`, and `run_sim`) as they existed when this
notebook was executed.

### Initial conditions used by the solver

In [8]:
print("Chosen calibration (Step 3):")
print(f"  bias_correction     = {chosen_mode!r}")
print(f"  evap_wind_floor_ms  = {chosen_wind_floor} m/s")
print()

print("Pool physical parameters (Waco, scripts/pools_config.json):")
print(f"  depth_m             = {depth_m} m")
print(f"  ground_temp_C       = {ground_temp_C} °C  ({C_to_F(ground_temp_C):.1f} °F)")
print(f"  k_soil_Wm1K         = {k_soil_Wm1K} W/m/K")
print(f"  soil_depth_m        = {soil_depth_m} m")
print(f"  K_CONCRETE          = {K_CONCRETE} W/m/K")
print(f"  L_CONCRETE_M        = {L_CONCRETE_M} m")
print(f"  R_concrete          = {R_concrete:.4f} m²K/W")
print(f"  R_soil              = {R_soil:.4f} m²K/W")
print(f"  bottom_u_Wm2K       = {bottom_u_Wm2K:.4f} W/m²/K  (derived, not set directly)")
print()

print("Production defaults from pool_physics.py (for comparison):")
print(f"  CE_MULTIPLIER       = {CE_MULTIPLIER}")
print(f"  EVAP_WIND_FLOOR_MS  = {EVAP_WIND_FLOOR_MS} m/s")
print()

print("Simulation harness (run_sim) settings used in Step 3:")
print(f"  T_pool_start_C      = 30.0 °C  ({C_to_F(30.0):.1f} °F)  (arbitrary seed — drift_C confirms it washes out)")
print(f"  days                = 20  (run to steady periodic state)")
print(f"  dt_days             = 1/24  (hourly time step)")
print()

print("diurnal_forcing() default anchors (Waco humid-summer scenario):")
print(f"  day_high_C  = {F_to_C(110):.2f} °C  (110 °F)")
print(f"  night_low_C = {F_to_C(80):.2f} °C  (80 °F)")
print(f"  rh_day      = 55.0 %")
print(f"  rh_night    = 80.0 %")
print(f"  wind_day    = 2.6 m/s")
print(f"  wind_night  = 1.4 m/s")
print(f"  solar_peak  = 980.0 W/m²")

Chosen calibration (Step 3):
  bias_correction     = 'wind_floor'
  evap_wind_floor_ms  = 14.0 m/s

Pool physical parameters (Waco, scripts/pools_config.json):
  depth_m             = 2.0 m
  ground_temp_C       = 20.0 °C  (68.0 °F)
  k_soil_Wm1K         = 1.5 W/m/K
  soil_depth_m        = 2.0 m
  K_CONCRETE          = 1.4 W/m/K
  L_CONCRETE_M        = 0.3048 m
  R_concrete          = 0.2177 m²K/W
  R_soil              = 1.3333 m²K/W
  bottom_u_Wm2K       = 0.6447 W/m²/K  (derived, not set directly)

Production defaults from pool_physics.py (for comparison):
  CE_MULTIPLIER       = 1.25
  EVAP_WIND_FLOOR_MS  = 1.5 m/s

Simulation harness (run_sim) settings used in Step 3:
  T_pool_start_C      = 30.0 °C  (86.0 °F)  (arbitrary seed — drift_C confirms it washes out)
  days                = 20  (run to steady periodic state)
  dt_days             = 1/24  (hourly time step)

diurnal_forcing() default anchors (Waco humid-summer scenario):
  day_high_C  = 43.33 °C  (110 °F)
  night_low_C = 2

### Two days of hourly forcing fed into the solver

`run_sim` calls `diurnal_forcing(h + 0.5)` every hour — i.e. each hourly
step uses the mid-hour value of the diurnal curve. Below are the first
48 hourly steps (days 1–2) exactly as the solver consumes them.

In [9]:
print(f"{'day':>3s} {'hour':>4s} {'T_air_F':>8s} {'RH_%':>6s} {'wind_ms':>8s} {'solar_MJm2day':>14s}")
for day in (1, 2):
    for h in range(24):
        T_air, RH, wind, solar = diurnal_forcing(h + 0.5)
        print(f"{day:3d} {h:4d} {C_to_F(T_air):8.1f} {RH:6.1f} {wind:8.2f} {solar:14.2f}")

day hour  T_air_F   RH_%  wind_ms  solar_MJm2day
  1    0     93.0   69.1     1.40           0.00
  1    1     89.3   72.3     1.40           0.00
  1    2     85.9   75.1     1.40           0.00
  1    3     83.1   77.4     1.40           0.00
  1    4     81.1   79.0     1.40           0.00
  1    5     80.1   79.9     1.40           0.00
  1    6     80.1   79.9     1.56           9.48
  1    7     81.1   79.0     1.86          27.97
  1    8     83.1   77.4     2.13          45.05
  1    9     85.9   75.1     2.35          59.87
  1   10     89.3   72.3     2.51          71.69
  1   11     93.0   69.1     2.59          79.92
  1   12     97.0   65.9     2.59          84.14
  1   13    100.7   62.7     2.51          84.14
  1   14    104.1   59.9     2.35          79.92
  1   15    106.9   57.6     2.13          71.69
  1   16    108.9   56.0     1.86          59.87
  1   17    109.9   55.1     1.56          45.05
  1   18    109.9   55.1     1.40          27.97
  1   19    108.9   

### Entire solver source

Full source of every function the simulation calls: the forcing curve
(`diurnal_forcing`), the multi-day harness (`run_sim`), and the underlying
energy-balance step (`pool_thermal_balance_step`, imported from
`pool_physics`).

In [10]:
import inspect
import pool_physics

for fn in (diurnal_forcing, run_sim, pool_physics.pool_thermal_balance_step):
    print(f"# {'=' * 74}")
    print(f"# {fn.__module__}.{fn.__qualname__}")
    print(f"# {'=' * 74}")
    print(inspect.getsource(fn))
    print()

# ==========================================================================
# __main__.diurnal_forcing
# ==========================================================================
def diurnal_forcing(hour,
                     day_high_C=F_to_C(110), night_low_C=F_to_C(80),
                     rh_day=55.0, rh_night=80.0,
                     wind_day=2.6, wind_night=1.4,
                     solar_peak_Wm2=980.0):
    """Smooth diurnal cycle for one hour-of-day (0-24, fractional OK)."""
    frac = (hour - 6.0) / 24.0
    T_air = ((day_high_C + night_low_C) / 2
             + (day_high_C - night_low_C) / 2 * math.cos(2 * math.pi * (frac - 0.5)))

    # RH tracks inversely with temperature (dry heat of the day, humid at night)
    rh_frac = (T_air - night_low_C) / (day_high_C - night_low_C)
    RH = rh_day + (rh_night - rh_day) * (1 - rh_frac)

    # Wind picks up slightly at midday, calmer overnight
    if 6.0 <= hour <= 18.0:
        wind = wind_night + (wind_day - wind_night) * max(

## Head-to-Head: Production Code vs. Calibrated Corrections, Across Conditions

Everything above was tuned against a single scenario (Waco humid summer,
110°F day / 80°F night). Before recommending a change, check how the
**current operational code** (`bias_correction='none'`, i.e.
`ACTIVE_BIAS_CORRECTION` as shipped in `pool_physics.py` — no adjustment)
compares against the two calibrations found above
(`evap_multiplier` @ `CE_MULTIPLIER=8`, `wind_floor` @
`EVAP_WIND_FLOOR_MS=14 m/s`) across a **variety of conditions**, not just
the one it was fit to.

Same Waco physical parameters (`depth_m`, `ground_temp_C`, `bottom_u_Wm2K`)
are held fixed across all scenarios below so the only thing varying is the
weather forcing — this isolates how each correction *generalizes*, rather
than mixing in pool-to-pool physical differences.

In [11]:
scenarios = {
    'Waco humid summer (calibration scenario)': dict(
        day_high_C=F_to_C(110), night_low_C=F_to_C(80),
        rh_day=55.0, rh_night=80.0,
        wind_day=2.6, wind_night=1.4,
        solar_peak_Wm2=980.0,
    ),
    'Desert dry summer (Palm Springs-like)': dict(
        day_high_C=F_to_C(115), night_low_C=F_to_C(78),
        rh_day=15.0, rh_night=35.0,
        wind_day=2.4, wind_night=1.2,
        solar_peak_Wm2=1050.0,
    ),
    'Coastal humid summer (Virginia Beach-like)': dict(
        day_high_C=F_to_C(92), night_low_C=F_to_C(76),
        rh_day=65.0, rh_night=85.0,
        wind_day=4.0, wind_night=2.5,
        solar_peak_Wm2=850.0,
    ),
    'Mild spring (Lemoore-like)': dict(
        day_high_C=F_to_C(85), night_low_C=F_to_C(55),
        rh_day=35.0, rh_night=60.0,
        wind_day=3.0, wind_night=1.0,
        solar_peak_Wm2=900.0,
    ),
    'Overcast / cool day': dict(
        day_high_C=F_to_C(78), night_low_C=F_to_C(62),
        rh_day=70.0, rh_night=85.0,
        wind_day=2.0, wind_night=1.5,
        solar_peak_Wm2=500.0,
    ),
}

for label, kwargs in scenarios.items():
    print(f"{label}:")
    print(f"  day_high={C_to_F(kwargs['day_high_C']):.0f}F  night_low={C_to_F(kwargs['night_low_C']):.0f}F  "
          f"RH={kwargs['rh_day']:.0f}/{kwargs['rh_night']:.0f}%  "
          f"wind={kwargs['wind_day']:.1f}/{kwargs['wind_night']:.1f} m/s  "
          f"solar_peak={kwargs['solar_peak_Wm2']:.0f} W/m²")

Waco humid summer (calibration scenario):
  day_high=110F  night_low=80F  RH=55/80%  wind=2.6/1.4 m/s  solar_peak=980 W/m²
Desert dry summer (Palm Springs-like):
  day_high=115F  night_low=78F  RH=15/35%  wind=2.4/1.2 m/s  solar_peak=1050 W/m²
Coastal humid summer (Virginia Beach-like):
  day_high=92F  night_low=76F  RH=65/85%  wind=4.0/2.5 m/s  solar_peak=850 W/m²
Mild spring (Lemoore-like):
  day_high=85F  night_low=55F  RH=35/60%  wind=3.0/1.0 m/s  solar_peak=900 W/m²
Overcast / cool day:
  day_high=78F  night_low=62F  RH=70/85%  wind=2.0/1.5 m/s  solar_peak=500 W/m²


In [12]:
def run_sim_scenario(mode, forcing_kwargs, days=20, T_pool_start_C=30.0,
                      ce_multiplier=None, evap_wind_floor_ms=None):
    """Same solver/harness as run_sim, but forcing_kwargs plug into diurnal_forcing
    so the same code path can be driven by any of the scenarios above."""
    T_pool = T_pool_start_C
    dt = 1.0 / 24.0
    prev_day, last_day = None, None

    for _ in range(days):
        hourly = []
        for h in range(24):
            T_air, RH, wind, solar = diurnal_forcing(h + 0.5, **forcing_kwargs)
            T_pool, fx = pool_thermal_balance_step(
                T_pool, T_air, RH, wind, solar,
                depth_m=depth_m, ground_temp_C=ground_temp_C, bottom_u_Wm2K=bottom_u_Wm2K,
                dt_days=dt, bias_correction=mode,
                ce_multiplier=ce_multiplier, evap_wind_floor_ms=evap_wind_floor_ms,
            )
            hourly.append(T_pool)
        prev_day, last_day = last_day, hourly

    drift_C = max(abs(a - b) for a, b in zip(last_day, prev_day)) if prev_day else float('nan')
    return {
        'hourly_C': last_day,
        'mean_C': sum(last_day) / len(last_day),
        'min_C': min(last_day),
        'max_C': max(last_day),
        'drift_C': drift_C,
    }

# Head-to-head configurations: production default vs. the two calibrations from Step 2/3
CONFIGS = [
    ('current (none, no adjustment)', dict(mode='none')),
    ('evap_multiplier (CE=8)',        dict(mode='evap_multiplier', ce_multiplier=8.0)),
    ('wind_floor (14 m/s)',           dict(mode='wind_floor', evap_wind_floor_ms=14.0)),
]

results_by_scenario = {}
for label, forcing_kwargs in scenarios.items():
    results_by_scenario[label] = {
        cfg_label: run_sim_scenario(forcing_kwargs=forcing_kwargs, **cfg_kwargs)
        for cfg_label, cfg_kwargs in CONFIGS
    }

print(f"Ran {len(CONFIGS)} configs × {len(scenarios)} scenarios = "
      f"{len(CONFIGS) * len(scenarios)} simulations.")

Ran 3 configs × 5 scenarios = 15 simulations.


In [13]:
for label in scenarios:
    print(f"=== {label} ===")
    print(f"{'config':32s} {'mean':>8s} {'min':>8s} {'max':>8s} {'drift_C':>10s}")
    baseline_mean_F = C_to_F(results_by_scenario[label]['current (none, no adjustment)']['mean_C'])
    for cfg_label, _ in CONFIGS:
        r = results_by_scenario[label][cfg_label]
        mean_F = C_to_F(r['mean_C'])
        delta_str = '' if cfg_label.startswith('current') else f"  (Δ{mean_F - baseline_mean_F:+.1f}F vs current)"
        print(f"{cfg_label:32s} {mean_F:7.1f}F {C_to_F(r['min_C']):7.1f}F {C_to_F(r['max_C']):7.1f}F {r['drift_C']:10.4f}{delta_str}")
    print()

=== Waco humid summer (calibration scenario) ===
config                               mean      min      max    drift_C
current (none, no adjustment)      107.0F   105.4F   108.6F     0.0038
evap_multiplier (CE=8)              90.7F    88.4F    92.6F     0.0000  (Δ-16.4F vs current)
wind_floor (14 m/s)                 90.5F    88.0F    92.7F     0.0000  (Δ-16.5F vs current)

=== Desert dry summer (Palm Springs-like) ===
config                               mean      min      max    drift_C
current (none, no adjustment)      102.4F   100.7F   104.0F     0.0063
evap_multiplier (CE=8)              75.9F    74.2F    77.4F     0.0000  (Δ-26.5F vs current)
wind_floor (14 m/s)                 74.5F    72.3F    76.4F     0.0000  (Δ-27.9F vs current)

=== Coastal humid summer (Virginia Beach-like) ===
config                               mean      min      max    drift_C
current (none, no adjustment)       92.0F    90.6F    93.3F     0.0004
evap_multiplier (CE=8)              80.3F    78.6F    

In [14]:
print("Cross-scenario summary — mean equilibrium pool temp (°F) per config:\n")
header = f"{'scenario':42s}" + ''.join(f"{cfg_label[:20]:>22s}" for cfg_label, _ in CONFIGS)
print(header)
for label in scenarios:
    row = f"{label:42s}"
    for cfg_label, _ in CONFIGS:
        mean_F = C_to_F(results_by_scenario[label][cfg_label]['mean_C'])
        row += f"{mean_F:22.1f}"
    print(row)

Cross-scenario summary — mean equilibrium pool temp (°F) per config:

scenario                                    current (none, no ad  evap_multiplier (CE=   wind_floor (14 m/s)
Waco humid summer (calibration scenario)                   107.0                  90.7                  90.5
Desert dry summer (Palm Springs-like)                      102.4                  75.9                  74.5
Coastal humid summer (Virginia Beach-like)                  92.0                  80.3                  82.6
Mild spring (Lemoore-like)                                  83.1                  65.2                  64.4
Overcast / cool day                                         77.1                  68.5                  68.3


### Result — the calibration does not generalize

The `CE_MULTIPLIER=8` / `EVAP_WIND_FLOOR_MS=14 m/s` calibration only looks
good on the exact scenario it was fit to (Waco humid summer, ~90.5-90.7°F).
Applied unchanged to other conditions, both corrections **over-cool
substantially** as the day gets less hot/humid:

| scenario | current (none) | evap_multiplier | wind_floor |
|---|---|---|---|
| Waco humid summer (fit target) | 107.0°F | 90.7°F | 90.5°F |
| Desert dry summer | 102.4°F | 75.9°F | 74.5°F |
| Coastal humid summer | 92.0°F | 80.3°F | 82.6°F |
| Mild spring | 83.1°F | 65.2°F | 64.4°F |
| Overcast / cool day | 77.1°F | 68.5°F | 68.3°F |

Both corrections apply a **fixed** extra evaporative-cooling penalty
(`CE_MULTIPLIER` scales the coefficient; `EVAP_WIND_FLOOR_MS` floors the
effective wind) regardless of how hot/humid the day actually is. That's a
reasonable fix for the specific failure mode this was built for (baseline
under-cools on hot/calm/sunny days), but it is **not conditioned on
temperature or humidity**, so on a mild 85°F day it still subtracts
roughly the same evaporative penalty it applied on a 110°F day — pushing
a already-plausible ~83°F baseline down to an implausible mid-60s°F.

**Conclusion**: neither `CE_MULTIPLIER=8` nor `EVAP_WIND_FLOOR_MS=14 m/s`
should replace the production default (`ACTIVE_BIAS_CORRECTION='none'`)
as a flat, always-on constant. Any future bias correction needs to scale
with conditions (e.g. only engage the extra cooling once air temp exceeds
some threshold, or make the correction proportional to the baseline
model's own under-cooling residual) rather than applying a single
magnitude tuned to one hot-day scenario across the whole year.

## Temperature-Ramped Correction — scale the fix with how hot it actually is

Instead of a flat `CE_MULTIPLIER=8` / `EVAP_WIND_FLOOR_MS=14` applied on
every single hour regardless of conditions, scale the correction magnitude
continuously with the instantaneous air temperature for that hour:

- Below `T_low` (no extra cooling needed — baseline is already plausible
  there per the head-to-head results above): use the **production
  default** magnitude (`CE_MULTIPLIER=1.25` / `EVAP_WIND_FLOOR_MS=1.5`,
  i.e. effectively no correction).
- At/above `T_high` (the Waco calibration scenario): use the **full**
  calibrated magnitude (`CE_MULTIPLIER=8` / `EVAP_WIND_FLOOR_MS=14`).
- Between the two, interpolate with a smoothstep curve
  (`3w² − 2w³`, `w = (T_air − T_low)/(T_high − T_low)`) rather than a
  straight line, so the correction ramps in gradually with no kink in the
  derivative (a hard corner would let the forward-Euler solver see abrupt
  jumps in `Q_evap` hour-to-hour).

`T_low = 85°F` and `T_high = 110°F` are chosen from the head-to-head
findings: 85°F is roughly the "mild spring" scenario where the
uncorrected baseline (83.1°F) was already plausible, and 110°F is the
Waco calibration target this was tuned against.

**Acceptance band, not a single target**: the "right" answer isn't a
strict ~90°F — up to **92°F is also reasonable** for scenarios that are
humid, sunny, *and* air temp exceeds 110°F (i.e. hotter than the
calibration scenario itself). So when scoring the ramped correction below,
treat **88–92°F** as the acceptable range at/above `T_high`, rather than
penalizing anything over 90°F.

In [ ]:
T_LOW_F, T_HIGH_F = 85.0, 110.0
T_LOW_C, T_HIGH_C = F_to_C(T_LOW_F), F_to_C(T_HIGH_F)

def temp_ramp(T_air_C, T_low_C=T_LOW_C, T_high_C=T_high_C if False else T_HIGH_C):
    """0→1 smoothstep weight for how far T_air is between T_low and T_high."""
    if T_high_C <= T_low_C:
        raise ValueError("T_high_C must exceed T_low_C")
    w = (T_air_C - T_low_C) / (T_high_C - T_low_C)
    w = min(1.0, max(0.0, w))
    return w * w * (3 - 2 * w)  # smoothstep

# Diagnostic: show the resulting magnitude at a spread of air temps
print(f"{'T_air_F':>8s} {'ramp_w':>7s} {'CE_MULTIPLIER':>14s} {'wind_floor_ms':>14s}")
for T_air_F in [70, 75, 80, 85, 90, 95, 100, 105, 110, 115]:
    w = temp_ramp(F_to_C(T_air_F))
    ce = CE_MULTIPLIER + (8.0 - CE_MULTIPLIER) * w
    wf = EVAP_WIND_FLOOR_MS + (14.0 - EVAP_WIND_FLOOR_MS) * w
    print(f"{T_air_F:8.0f} {w:7.3f} {ce:14.2f} {wf:14.2f}")